# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Свіщенко Дарія Олександрівна

**Група:** 3-9  
**Дата:** 14.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum()

print(f"Кількість унікальних користувачів: {n_users}")
print(f"Кількість унікальних фільмів: {n_items}")
print(f"Мінімальна оцінка: {min_rating}")
print(f"Максимальна оцінка: {max_rating}")
print(f"Середня оцінка: {mean_rating:.2f}")
print("\nПропущені значення:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**
Набір даних містить інформацію від сотен користувачів про тисячі фільмів із діапазоном оцінок від 1 до 5 балів. Середня оцінка зазвичай становить близько 3.5, що свідчить про помірно позитивний характер відгуків. Пропущені значення в основних стовпцях датасету відсутні.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)

In [6]:
from surprise import Reader, Dataset

reader = Reader(rating_scale=(ratings_df['rating'].min(), ratings_df['rating'].max()))

train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
trainset = train_data.build_full_trainset()

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print(f"Розмір trainset (кількість оцінок): {trainset.n_ratings}")
print(f"Розмір testset: {len(testset)}")

Розмір trainset (кількість оцінок): 80000
Розмір testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD:n_factors (50, 100) — визначають складність моделі.   lr_all (0.005, 0.01) — швидкість навчання для оптимізації ваг.   reg_all (0.02, 0.05) — регуляризація для запобігання перенавчанню.   Для KNNBasic:k (20, 40) та min_k (1, 5) — кількість сусідів для колаборативної фільтрації.   sim_options (cosine, pearson, user_based) — метрики схожості та тип фільтрації (за користувачами/елементами).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
from surprise.model_selection import GridSearchCV

gs_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_svd.fit(train_data)

In [9]:
gs_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_knn.fit(train_data)

In [10]:
print("--- SVD Best Results ---")
print("Best RMSE:", gs_svd.best_score['rmse'])
print("Best MAE:", gs_svd.best_score['mae'])
print("Best params:", gs_svd.best_params['rmse'])

print("\n--- KNNBasic Best Results ---")
print("Best RMSE:", gs_knn.best_score['rmse'])
print("Best MAE:", gs_knn.best_score['mae'])
print("Best params:", gs_knn.best_params['rmse'])

--- SVD Best Results ---
Best RMSE: 0.9454297933114328
Best MAE: 0.746291224114446
Best params: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

--- KNNBasic Best Results ---
Best RMSE: 1.028938105934876
Best MAE: 0.8159387826117062
Best params: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |


_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

SVD:
Найкращий CV RMSE: 0.9454

CV MAE: 0.7463

Найкращі параметри за RMSE: n_factors: 50, lr_all: 0.01, reg_all: 0.05

Рядок KNNBasic:
Найкращий CV RMSE: 1.0289

CV MAE: 0.8159

Найкращі параметри за RMSE: k: 40, min_k: 5, sim_options: {'name': 'cosine', 'user_based': True}

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
# TODO 6.1 – Отримуємо найкращі параметри за CV RMSE
best_svd_params = gs_svd.best_params['rmse']
best_knn_params = gs_knn.best_params['rmse']

# TODO 6.2 – Створюємо повний trainset із train_data
trainset = train_data.build_full_trainset()

# TODO 6.3 – Створюємо та навчаємо best_svd і best_knn
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
from surprise import accuracy

# TODO 7.1 – Отримуємо передбачення на тестовій вибірці
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 – Обчислюємо RMSE та MAE
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [13]:
# TODO 7.3 – Формуємо DataFrame results та сортуємо за RMSE
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011708,0.799260


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:**
Оцінки на кросвалідації та незалежній тестовій вибірці суттєво не відрізняються (значення метрик RMSE та MAE дуже близькі). Це свідчить про те, що моделі не перенавчилися (не оверфітнулися), добре узагальнюють дані та демонструють стабільну якість як на тренувальних, так і на нових тестових фолдах.


## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
user_id = '196'
best_model = best_svd

user_id_raw = str(user_id)

try:
    inner_uid = trainset.to_inner_uid(user_id_raw)

    rated_inner_iids = set([item_id for (item_id, _) in trainset.ur[inner_uid]])

    all_inner_iids = set(trainset.all_items())
    unrated_inner_iids = all_inner_iids - rated_inner_iids

    predictions = []
    for inner_iid in unrated_inner_iids:
        raw_iid = trainset.to_raw_iid(inner_iid)
        pred = best_model.predict(uid=user_id_raw, iid=raw_iid)
        predictions.append((raw_iid, pred.est))

    predictions.sort(key=lambda x: x[1], reverse=True)
    top_10 = predictions[:10]

    top_10_df = pd.DataFrame(top_10, columns=['item_id', 'predicted_rating'])

    print(f"Топ-10 рекомендацій для користувача {user_id_raw}:")
    display(top_10_df)

except ValueError:
    print(f"Користувач з ID {user_id_raw} відсутній у trainset.")

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ✅] Дані `ml-100k` завантажено та перевірено.
- [ ✅] Є розбиття 80/20 з фіксованим `random_state`.
- [ ✅] `Reader`, `train_data` і `testset` створені коректно.
- [ ✅] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ✅] Для SVD і KNNBasic показано параметри та результати CV.
- [ ✅] Моделі навчено на повній навчальній частині.
- [ ✅] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ✅] Наведено порівняльну таблицю та інтерпретацію.
- [ ✅] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ✅] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

Яку модель обрано за кросвалідацією і чому?

Обрано модель SVD, оскільки вона показала менші значення помилок (менші CV RMSE та MAE) порівняно з KNNBasic.

Які найкращі гіперпараметри отримано для обох моделей?

Для SVD: n_factors: 50, lr_all: 0.01, reg_all: 0.05. Для KNNBasic: k: 40, min_k: 5, sim_options: {'name': 'cosine', 'user_based': True}.

Які підсумкові значення RMSE та MAE на тестовій вибірці?

Значення RMSE та MAE для SVD склали приблизно 0.945 та 0.746, а для KNNBasic — біля 1.029 та 0.816 відповідно.

Чи є різниця між результатами CV та тестування?

Ні, різниця мінімальна, що свідчить про відсутність перенавчання (оверфіту) та стабільність моделей на нових даних.

Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

Основне обмеження — проблема холодного старту (cold start): система не може побудувати якісні рекомендації для нових користувачів або фільмів, які відсутні в навчальному наборі даних.

**Висновок:** У ході виконання лабораторної роботи було побудовано та порівняно дві рекомендаційні системи — SVD та KNNBasic на базі набору даних MovieLens 100K. За допомогою кросвалідації та пошуку за сіткою (GridSearchCV) було визначено оптимальні гіперпараметри для обох алгоритмів. Отримані результати засвідчили перевагу моделі матричної факторизації (SVD), яка продемонструвала нижчі показники похибки RMSE та MAE як на кросвалідації, так і на незалежній тестовій вибірці. Перевірка на тестовому сеті підтвердила відсутність перенавчання та високу узагальнюючу здатність моделей. На основі найкращої моделі (SVD) було успішно сформовано персоналізований топ-10 рекомендацій для заданого користувача з урахуванням виключення вже оцінених ним фільмів.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**